# Lab 06 — Class Imbalance & Cost-Sensitive Learning

**Research question:** What happens when attacks become rare in the training data?

We keep the official test set untouched and compare:
- no class weighting;
- balanced class weighting;
- controlled random undersampling.

In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame
from src.imbalance import (
    class_balance_frame,
    make_controlled_binary_imbalance,
    random_undersample_majority,
    balanced_class_weight_dict,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
X_imbalanced, y_imbalanced = make_controlled_binary_imbalance(
    X_train,
    y_train,
    positive_to_negative_ratio=0.10,
    random_state=42,
)

print("Original:")
display(class_balance_frame(y_train))

print("Controlled rare-attack training set:")
display(class_balance_frame(y_imbalanced))

print("Balanced class weights:")
print(balanced_class_weight_dict(y_imbalanced))

In [ ]:
unweighted = logistic_pipeline(
    X_imbalanced,
    class_weight=None,
)
unweighted.fit(X_imbalanced, y_imbalanced)

pred_unweighted = unweighted.predict(X_test)
score_unweighted = unweighted.predict_proba(X_test)[:, 1]
m_unweighted = binary_metrics(
    y_test,
    pred_unweighted,
    score_unweighted,
)

In [ ]:
weighted = logistic_pipeline(
    X_imbalanced,
    class_weight="balanced",
)
weighted.fit(X_imbalanced, y_imbalanced)

pred_weighted = weighted.predict(X_test)
score_weighted = weighted.predict_proba(X_test)[:, 1]
m_weighted = binary_metrics(
    y_test,
    pred_weighted,
    score_weighted,
)

In [ ]:
X_under, y_under = random_undersample_majority(
    X_imbalanced,
    y_imbalanced,
    target_ratio=0.50,
    random_state=42,
)

undersampled = logistic_pipeline(
    X_under,
    class_weight=None,
)
undersampled.fit(X_under, y_under)

pred_under = undersampled.predict(X_test)
score_under = undersampled.predict_proba(X_test)[:, 1]
m_under = binary_metrics(
    y_test,
    pred_under,
    score_under,
)

In [ ]:
comparison = pd.concat([
    binary_metrics_frame("Unweighted", m_unweighted),
    binary_metrics_frame("Class weighted", m_weighted),
    binary_metrics_frame("Undersampled", m_under),
])

display(
    comparison[
        [
            "precision",
            "recall",
            "f1",
            "false_positive_rate",
            "false_negative_rate",
            "roc_auc",
        ]
    ].round(4)
)

## Explain the result

- Which method recovered attack recall?
- What happened to precision?
- Which method had the lowest false-negative rate?
- What information did undersampling discard?
- Why must balancing occur only inside training data?